# Weights & Biases Automated Hyperparameter Optimization 
**EGR 444: Advanced AI/ML for Engineering Applications**

By the end of the workshop, we will configure and run a Weights & Biases (W&B) Sweep on a baseline PyTorch neural network. We will produce an interactive sweep dashboard containing parallel coordinate plots, parameter importance rankings, and real-time loss/validation curves across 3 automated trials.

We will work with **Fashion-MNIST**, created by Zalando Research and released under the MIT License. It consists of 70,000 28×28 grayscale images across 10 clothing and footwear categories (such as T-shirts/tops, trousers, sneakers, and ankle boots) with single-class labels. We preserve the standard 60,000-image training and 10,000-image test split.

Some of the code is adapted from the official Pytorch documentation at this link:
https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html

**Local setup:** use Python 3.12, install the requirements, and select that environment as your notebook kernel.

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import v2
import matplotlib.pyplot as plt

# Download train and test datasets
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

# Plot random samples to learn about the data

Verifies that images and labels are loaded and aligned correctly before training starts. Visual inspection catches bugs and nuances within the data like wrong image dimensions or mismatched class indices.

In [ ]:
# Create dictionary with dataset labels for plotting
labels_map = {
    0: "T-Shirt",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle Boot",
}

# Create plot
figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3
for i in range(1, cols * rows + 1):
    # Plot random images from the dataset
    sample_idx = torch.randint(len(training_data), size=(1,)).item()
    img, label = training_data[sample_idx]
    figure.add_subplot(rows, cols, i)
    plt.title(labels_map[label])
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")
plt.show()

# Create custom dataset class

Standardizes how raw images and labels are indexed, transformed, and returned in a uniform format.

In [ ]:
import os
import pandas as pd
from torchvision.io import decode_image

# Create dataset class wrapper for the dataset
class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = decode_image(img_path)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

# Create dataloaders 

Wraps the dataset to handle mini-batching, shuffling, and data loading.

In [ ]:
from torch.utils.data import DataLoader

# Create example dataloaders for plotting
train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=True)

# Display random image and label
train_features, train_labels = next(iter(train_dataloader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")
img = train_features[0].squeeze()
label = train_labels[0]
plt.imshow(img, cmap="gray")
plt.show()
print(f"Label: {label}\nClass name: {labels_map[label.item()]}")

# Set up W&B connection

To use W&B, you need to create an account at: https://wandb.ai/site/

After creating an account, you'll need to set up an API key.

In [ ]:
import wandb

# Connect W&B account to your notebook
wandb.login()

# Create your model weights output directory

In [ ]:
from pathlib import Path

# Create directory to save model weights
save_dir = Path("outputs/")
save_dir.mkdir(parents=True, exist_ok=True)

# Basic example without W&B Sweeps

This example shows you how to create a config variable and setup for your W&B run. We'll also walk through the W&B UI and some key features.

In [ ]:
from models import NeuralNetwork
import torch.nn as nn

ENTITY_NAME = "INSERT_YOUR_TEAM_NAME"
PROJECT_NAME = "egr444-wandb-workshop"
DEFAULT_CONFIG = {
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs": 10,
    "optimizer": "adam",
    "architecture": "NN",
    "dataset": "Fashion-MNIST",
}

# Start a new wandb run to track this script
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name)
    entity=ENTITY_NAME,
    # Set the wandb project where this run will be logged
    project=PROJECT_NAME,
    # Create default config hyperparameters
    config=DEFAULT_CONFIG,
)

# Read active hyperparameters from wandb.config
cfg = wandb.config
device = "cpu"

# Create dataloaders using wandb.config
train_dataloader = DataLoader(training_data, batch_size=cfg.batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=cfg.batch_size, shuffle=True)

# Instantiate model, loss, and optimizer using config values
model = NeuralNetwork().to(device)
criterion = nn.CrossEntropyLoss()

if cfg.optimizer == "adam":
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
else:
    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.learning_rate)

# Training loop
for epoch in range(cfg.epochs):
    # Training step
    model.train()
    running_train_loss = 0.0
    for X, y in train_dataloader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * X.size(0)

    train_loss = running_train_loss / len(train_dataloader.dataset)

    # Testing step
    model.eval()
    running_test_loss = 0.0
    correct = 0
    with torch.no_grad():
        for X, y in test_dataloader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = criterion(logits, y)

            running_test_loss += loss.item() * X.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()

    test_loss = running_test_loss / len(test_dataloader.dataset)
    test_acc = correct / len(test_dataloader.dataset)

    # Save model weights
    model_weight_path = f"outputs/{run.name}.pth"
    torch.save(model.state_dict(), model_weight_path)

    # Log metrics to W&B per epoch
    run.log({
        "epoch": epoch + 1,
        "learning_rate:": cfg.learning_rate,
        "batch_size": cfg.batch_size,
        "train_loss": train_loss,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "model_path": model_weight_path
    })

# Finish the run and sync remaining data
run.finish()

# Basic example with W&B Sweeps

This example shows you how to run a hyperparameter search using W&B Sweeps and walks you through how to see the **Parameter Importance** chart. We'll create a sweep config and a train function.

To see the **Parameter Importance** chart:
1. In the Charts section, click the three dots on the side
2. Click Add Panels
3. Under Evaluation, click Parameter Importance

In [ ]:
# Define your hyperparameter tuning boundaries
sweep_config = {
    'method': 'bayes',
    'metric': {
        'name': 'test_loss',
        'goal': 'minimize'
    },
    'parameters': {
        'epochs': {'value': 5},
        'batch_size': {'values': [32, 64, 128]},
        'lr': {
            'distribution': 'log_uniform_values',
            'min': 1e-4,
            'max': 1e-2
        },
        'optimizer': {'value': "adam"}
    }
}

In [ ]:
# Define the training function that W&B will execute
def train():
    # Initialize the wandb run. This automatically receives hyperparameters from the agent.
    with wandb.init() as run:
        # Read active hyperparameters from wandb.config
        cfg = wandb.config
        device = "cpu"

        # Create dataloaders using wandb.config
        train_dataloader = DataLoader(training_data, batch_size=cfg.batch_size, shuffle=True)
        test_dataloader = DataLoader(test_data, batch_size=cfg.batch_size, shuffle=True)

        # Instantiate model, loss, and optimizer using config values
        model = NeuralNetwork().to(device)
        criterion = nn.CrossEntropyLoss()

        if cfg.optimizer == "adam":
            optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
        else:
            optimizer = torch.optim.SGD(model.parameters(), lr=cfg.lr)

        # Training loop
        best_test_acc = 0.0
        for epoch in range(cfg.epochs):
            # Training step
            model.train()
            running_train_loss = 0.0
            for X, y in train_dataloader:
                X, y = X.to(device), y.to(device)

                optimizer.zero_grad()
                logits = model(X)
                loss = criterion(logits, y)
                loss.backward()
                optimizer.step()

                running_train_loss += loss.item() * X.size(0)

            train_loss = running_train_loss / len(train_dataloader.dataset)

            # Testing step
            model.eval()
            running_test_loss = 0.0
            correct = 0
            with torch.no_grad():
                for X, y in test_dataloader:
                    X, y = X.to(device), y.to(device)
                    logits = model(X)
                    loss = criterion(logits, y)

                    running_test_loss += loss.item() * X.size(0)
                    preds = logits.argmax(dim=1)
                    correct += (preds == y).sum().item()

            test_loss = running_test_loss / len(test_dataloader.dataset)
            test_acc = correct / len(test_dataloader.dataset)

            # Save model weights
            model_weight_path = f"outputs/{run.name}.pth"
            # Save best checkpoint and log once per epoch
            if test_acc > best_test_acc:
                best_test_acc = test_acc
                torch.save(model.state_dict(), model_weight_path)

            # Log metrics to W&B
            run.log({
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "test_loss": test_loss,
                "test_acc": test_acc,
                "model_path": model_weight_path
            })

In [ ]:
# Initialize and launch the sweep
sweep_id = wandb.sweep(sweep=sweep_config, project=PROJECT_NAME)

# Launch the agent to run N variations
N_VARIATONS = 3
wandb.agent(sweep_id, function=train, count=N_VARIATONS)

# Adding a new hyperparameter to sweep

We're going to create a modified version of our baseline model to accept a variable hidden size.

In [ ]:
import torch.nn as nn

class VariableNeuralNetwork(nn.Module):
    def __init__(self, hidden_size=512):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

### Add hyperparameter to sweep config

Config method types to try:
* bayes
* random
* grid

What they do:
* bayes: Uses a Gaussian Process probabilistic model to predict which hyperparameter combinations will improve the target metric based on prior runs.
* random: Samples values independently from your defined parameter distributions.
* grid: Exhaustively tests every possible combination of discrete parameter lists.

When to use them:
* bayes: You have a small-to-medium set of continuous parameters (like learning_rate or dropout) and want efficient convergence within a fixed trial budget.
* random: You have many parameters, are doing an initial broad exploration, or want easy parallelization across multiple agents without sequential bottlenecks.
* grid: You have a very small, strictly discrete set of options (e.g., comparing 2 optimizers across 3 layer widths).

In [ ]:
# Define your hyperparameter tuning boundaries
sweep_config = {
    'method': 'bayes',                              # Values to try: 'random' or 'bayes'
    'metric': {
        'name': 'test_loss',
        'goal': 'minimize'
    },
    'parameters': {
        'epochs': {'value': 5},
        'batch_size': {'values': [32, 64, 128]},
        'lr': {
            'distribution': 'log_uniform_values',
            'min': 1e-4,
            'max': 1e-2
        },
        'optimizer': {'value': "adam"},
        
        # ADD NEW HYPERPARAMETER HERE
        'XXXX': {'values': [128, 256, 512, 1024]},
    }
}

In [ ]:
# Define the training function that W&B will execute
def train():
    # Initialize the wandb run. This automatically receives hyperparameters from the agent.
    with wandb.init() as run:
        # Read active hyperparameters from wandb.config
        cfg = wandb.config
        device = "cpu"

        # Create dataloaders using wandb.config
        train_dataloader = DataLoader(training_data, batch_size=cfg.batch_size, shuffle=True)
        test_dataloader = DataLoader(test_data, batch_size=cfg.batch_size, shuffle=True)

        # Instantiate model, loss, and optimizer using config values
        model = VariableNeuralNetwork(XXXX=cfg.XXXX).to(device)         # FILL IN THE BLANK HERE
        criterion = nn.CrossEntropyLoss()

        if cfg.optimizer == "adam":
            optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
        else:
            optimizer = torch.optim.SGD(model.parameters(), lr=cfg.lr)

        # Training loop
        best_test_acc = 0.0
        for epoch in range(cfg.epochs):
            # Training step
            model.train()
            running_train_loss = 0.0
            for X, y in train_dataloader:
                X, y = X.to(device), y.to(device)

                optimizer.zero_grad()
                logits = model(X)
                loss = criterion(logits, y)
                loss.backward()
                optimizer.step()

                running_train_loss += loss.item() * X.size(0)

            train_loss = running_train_loss / len(train_dataloader.dataset)

            # Testing step
            model.eval()
            running_test_loss = 0.0
            correct = 0
            with torch.no_grad():
                for X, y in test_dataloader:
                    X, y = X.to(device), y.to(device)
                    logits = model(X)
                    loss = criterion(logits, y)

                    running_test_loss += loss.item() * X.size(0)
                    preds = logits.argmax(dim=1)
                    correct += (preds == y).sum().item()

            test_loss = running_test_loss / len(test_dataloader.dataset)
            test_acc = correct / len(test_dataloader.dataset)

            # Save model weights
            model_weight_path = f"outputs/{run.name}.pth"
            # Save best checkpoint and log once per epoch
            if test_acc > best_test_acc:
                best_test_acc = test_acc
                torch.save(model.state_dict(), model_weight_path)

            # Log metrics to W&B
            run.log({
                "epoch": epoch + 1,

                # LOG NEW HYPERPARAMETER HERE
                "XXXX": cfg.XXXX,

                "train_loss": train_loss,
                "test_loss": test_loss,
                "test_acc": test_acc,
                "model_path": model_weight_path
            })

In [ ]:
# Initialize and launch the sweep
sweep_id = wandb.sweep(sweep=sweep_config, project=PROJECT_NAME)

# Launch the agent to run N variations
N_VARIATONS = 3
wandb.agent(sweep_id, function=train, count=N_VARIATONS)

# Task for the class

### Compare the parameter importance charts for both sweeps. 

### Check the most important hyperparameter for test_loss. Did you see any change? 

### If so explain why and if not, explain why not.